# ALFA probabilistic GPU v1 — Colab L4

Bu notebook dondurulmuş `four_dataset_probabilistic_gpu_v1` sözleşmesini çalıştırır: geçmiş 32 satırdan bir sonraki satır için kanal bazlı `mu` ve `sigma` tahmini, maskeli Gaussian NLL, 30 epoch ve yalnız normal train uçuşları. Sonuç görüldükten sonra feature, split, epoch, clip veya eşik değiştirilmez.

**ALFA sınırı:** corpus yalnız 54 uçuş ve 15 normal uçuş içerir; `split_00` optimizer rolünde 11 normal train, validation rolünde 2 normal uçuş vardır. GPU bu veri/session kıtlığını çözmez. Sonuç yalnız exploratory/development kanıtıdır; üretim veya genelleme iddiası değildir. Ayrı uçuş-modu modelleri bu v1 sözleşmesinin parçası değildir.

## 1. Drive'ı bağla ve sabit yolları kur

Drive'daki `/content/drive/MyDrive/bykr/four_dataset_probabilistic_v1_transfer` klasörüne `transfer_index.json`, kod ZIP'i ve ALFA dataset ZIP'ini yükleyin. Dosya adları, byte boyutları ve SHA-256 değerleri index'ten seçilir; ZIP'ler repo-relative yolları korur. Checkpoint ve raporlar ayrı Drive run klasörüne yazılır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATASET = 'alfa'
TRANSFER_DIR = Path('/content/drive/MyDrive/bykr/four_dataset_probabilistic_v1_transfer')
REPO_ROOT = Path('/content/four_dataset_probabilistic_gpu_v1')
RUN_DIR = Path('/content/drive/MyDrive/bykr/four_dataset_probabilistic_v1_runs') / f'{DATASET}_gpu_v1'
REPO_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not TRANSFER_DIR.is_dir():
    raise FileNotFoundError(f'Transfer klasörü yok: {TRANSFER_DIR}')
print('Dataset :', DATASET)
print('Repo    :', REPO_ROOT)
print('Run     :', RUN_DIR)

## 2. CUDA runtime'ını fail-loudly, L4 profilini bilgilendirici doğrula

CUDA yoksa çalışma başlamaz ve CPU fallback yapılmaz. L4 önerilen profildir; T4/A100 gibi başka bir CUDA GPU görülürse yalnız uyarı verilir.

In [ ]:
import platform
import torch
print('Python:', platform.python_version())
print('Torch :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA yok; Colab L4 GPU runtime seçin.')
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU:', gpu_name, '| VRAM GiB:', round(vram_gib, 2))
if 'L4' not in gpu_name.upper():
    print(f'UYARI: Önerilen L4 yerine {gpu_name} kullanılıyor; CUDA contract geçerli.')

## 3. Kod ve ALFA veri ZIP'lerini güvenli ve tekrar çalıştırılabilir biçimde aç

Her paketin SHA-256 değeri marker'a yazılır. Aynı runtime içinde aynı byte'lar tekrar açılmaz; paket değişirse yeniden açılır. Path-traversal içeren ZIP reddedilir.

In [ ]:
import hashlib
import json
import zipfile

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

index_path = TRANSFER_DIR / 'transfer_index.json'
if not index_path.is_file():
    raise RuntimeError(f'CONTRACT ERROR: transfer_index.json yok: {index_path}')
expected_index_sha256 = '1da9acaa509941dbc3ff3b6db07bccc50f63b4b13063d15debcd59388ed11af0'
observed_index_sha256 = sha256_file(index_path)
if observed_index_sha256 != expected_index_sha256:
    raise RuntimeError(f'CONTRACT ERROR: transfer_index.json SHA-256 uyuşmazlığı: {observed_index_sha256}')
print('INDEX SHA PASS:', observed_index_sha256[:12])
transfer_index = json.loads(index_path.read_text(encoding='utf-8'))
if transfer_index.get('candidate_namespace') != 'four_dataset_probabilistic_gpu_v1':
    raise RuntimeError('CONTRACT ERROR: transfer index namespace uyuşmuyor')
if transfer_index.get('archives_built') is not True:
    raise RuntimeError('CONTRACT ERROR: transfer index built archive bildirmiyor')
records = transfer_index.get('archives')
if not isinstance(records, list):
    raise RuntimeError('CONTRACT ERROR: transfer index archives listesi yok')
code_matches = [r for r in records if 'code_and_contract' in str(r.get('path', ''))]
data_matches = [r for r in records if DATASET in str(r.get('path', '')) and r not in code_matches]
if len(code_matches) != 1 or len(data_matches) != 1:
    raise RuntimeError(f'CONTRACT ERROR: archive seçimi tekil değil: code={code_matches}, data={data_matches}')
def verify_record(record):
    name = record.get('path')
    expected_sha = record.get('sha256')
    expected_bytes = record.get('bytes')
    if not isinstance(name, str) or Path(name).name != name:
        raise RuntimeError(f'CONTRACT ERROR: güvensiz archive adı: {name!r}')
    if not isinstance(expected_sha, str) or len(expected_sha) != 64 or not isinstance(expected_bytes, int):
        raise RuntimeError(f'CONTRACT ERROR: hash/byte kaydı eksik: {name}')
    archive = TRANSFER_DIR / name
    if not archive.is_file() or archive.stat().st_size != expected_bytes:
        raise RuntimeError(f'CONTRACT ERROR: archive yok veya byte boyutu yanlış: {archive}')
    observed = sha256_file(archive)
    if observed != expected_sha:
        raise RuntimeError(f'CONTRACT ERROR: SHA-256 uyuşmazlığı: {archive}')
    print('SHA PASS:', name, observed[:12])
    return archive, observed
verified = [verify_record(code_matches[0]), verify_record(data_matches[0])]
CODE_ZIP, DATA_ZIP = verified[0][0], verified[1][0]
def extract_verified(archive, observed, destination):
    marker_dir = destination / '.colab_extraction_markers'
    marker_dir.mkdir(parents=True, exist_ok=True)
    marker = marker_dir / f'{archive.name}.sha256'
    if marker.exists() and marker.read_text(encoding='ascii').strip() == observed:
        print('Hazır, atlandı:', archive.name, observed[:12])
        return
    root = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError(f'Güvensiz ZIP yolu: {member.filename}')
        bundle.extractall(destination)
    marker.write_text(observed, encoding='ascii')
    print('Açıldı:', archive.name, observed[:12])

for archive, observed in verified:
    extract_verified(archive, observed, REPO_ROOT)
runner_path = REPO_ROOT / 'scripts/four_dataset_probabilistic_gpu_v1_runner.py'
if not runner_path.is_file():
    raise FileNotFoundError(f'Kod ZIP repo-relative açılmadı; runner bulunamadı: {runner_path}')
print('Runner:', runner_path)

## 4. Minimal bağımlılıkları kur

Colab'ın CUDA uyumlu Torch kurulumu korunur; Torch yeniden kurulmaz.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'pyarrow', 'scikit-learn', 'scipy', 'matplotlib'],
               check=True)
print('Bağımlılıklar hazır; mevcut CUDA Torch korunuyor:', torch.__version__)

## 5. Dondurulmuş contract ve veri paketini doğrula

Bu hücre başarısızsa eğitime geçmeyin. Split, normal-only optimizer rolü, gerekli veri ve sözleşme kontrolleri runner tarafından yapılır.

In [ ]:
import os
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_ROOT)
RUNNER = REPO_ROOT / 'scripts/four_dataset_probabilistic_gpu_v1_runner.py'
def runner_command(action):
    return [sys.executable, str(RUNNER), action,
            '--repo-root', str(REPO_ROOT), '--dataset', DATASET,
            '--run-dir', str(RUN_DIR), '--device', 'cuda']
verify_cmd = runner_command('verify')
print(' '.join(verify_cmd))
subprocess.run(verify_cmd, check=True, env=env)
print('VERIFY PASS:', DATASET)

## 6. Eğitimi başlat veya Drive checkpoint'inden devam et

Aynı `RUN_DIR` ile yeniden çalıştırmak runner'ın atomik checkpoint'inden devam eder. Colab koparsa 1–6. adımları yeniden çalıştırın. Epoch sayısını veya başka frozen değeri sonuçlara bakarak değiştirmeyin.

In [ ]:
train_cmd = runner_command('train')
print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True, env=env)
print('TRAIN komutu tamamlandı:', RUN_DIR)

## 7. Checkpoint, history ve report durumunu incele

Bu hücre tensorları ekrana dökmez; bulunan checkpoint'in ilerleme alanlarını ve JSON raporlarını gösterir. Eğitim sürüyorsa checkpoint görünmesi, report'un henüz bulunmaması normaldir.

In [ ]:
import json

run_files = sorted(p for p in RUN_DIR.rglob('*') if p.is_file())
print('Run dosyaları:')
for path in run_files:
    print(' -', path.relative_to(RUN_DIR), f'({path.stat().st_size / 2**20:.2f} MiB)')

checkpoint_files = sorted(set(RUN_DIR.glob('*checkpoint*.pt')) | set(RUN_DIR.glob('*checkpoint*.pth')))
for path in checkpoint_files:
    payload = torch.load(path, map_location='cpu', weights_only=False)
    print('\nCHECKPOINT:', path.name)
    if isinstance(payload, dict):
        print('keys:', sorted(payload))
        for key in ('completed_epochs', 'epoch', 'epoch_index', 'batch_index', 'source_index', 'global_step', 'saved_at_utc'):
            if key in payload:
                print(f'{key}:', payload[key])
    else:
        print('type:', type(payload).__name__)

json_files = sorted(set(RUN_DIR.glob('*history*.json')) | set(RUN_DIR.glob('*report*.json')) | set(RUN_DIR.glob('*summary*.json')) | set(RUN_DIR.glob('*progress*.json')))
for path in json_files:
    print('\nJSON:', path.name)
    print(json.dumps(json.loads(path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
if not checkpoint_files and not json_files:
    print('Henüz checkpoint/history/report yok; train hücresinin çıktısını kontrol edin.')

## 8. Basit epoch-loss grafiği

History JSON veya checkpoint içindeki history alanından train/validation loss serileri bulunabildiği anda çizilir. Bu grafik parametre seçmek için kullanılmaz.

In [ ]:
import matplotlib.pyplot as plt

def history_records(value):
    if isinstance(value, list) and all(isinstance(item, dict) for item in value):
        return value
    if isinstance(value, dict):
        for key in ('history', 'epochs', 'training_history'):
            records = value.get(key)
            if isinstance(records, list) and all(isinstance(item, dict) for item in records):
                return records
    return []

payloads = []
for path in sorted(set(RUN_DIR.glob('*history*.json')) | set(RUN_DIR.glob('*report*.json'))):
    payloads.append(json.loads(path.read_text(encoding='utf-8')))
if checkpoint_files:
    payloads.append(torch.load(checkpoint_files[-1], map_location='cpu', weights_only=False))
records = next((items for value in payloads if (items := history_records(value))), [])
train_keys = ('mean_train_gaussian_nll', 'train_loss', 'train_nll', 'mean_train_nll', 'mean_weighted_gaussian_nll', 'loss')
val_keys = ('mean_validation_gaussian_nll', 'val_loss', 'validation_loss', 'val_nll', 'mean_val_nll')
def first_numeric(record, keys):
    for key in keys:
        value = record.get(key)
        if isinstance(value, (int, float)):
            return float(value)
    return None
epochs = [record.get('epoch', index + 1) for index, record in enumerate(records)]
train_loss = [first_numeric(record, train_keys) for record in records]
val_loss = [first_numeric(record, val_keys) for record in records]
if records and any(value is not None for value in train_loss + val_loss):
    plt.figure(figsize=(8, 4))
    if any(value is not None for value in train_loss):
        plt.plot(epochs, train_loss, marker='o', label='train')
    if any(value is not None for value in val_loss):
        plt.plot(epochs, val_loss, marker='o', label='validation')
    plt.xlabel('Epoch')
    plt.ylabel('Masked Gaussian NLL / loss')
    plt.title(f'{DATASET} probabilistic GPU v1')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
else:
    print('Çizilebilir epoch history henüz yok. Mevcut record anahtarları:', sorted(records[0]) if records else [])

## Sonuç okuma disiplini

`training_report.json` oluştuğunda magnitude diagnostic ve development metrikleri olduğu gibi raporlanır. Başarısız veya magnitude-dominated sonuç nedeniyle aynı v1 içinde epoch, feature, clip, split ya da eşik değiştirilmez. ALFA'nın küçük ve session-kısıtlı corpus olduğu sonuç yorumunda mutlaka korunur.